# Multi-Model YOLO Validation and Reporting

Run the `run_yolo_validation_report.py` script on multiple YOLO models, collect metrics, and compare results.


In [1]:

# 1. Set Up Environment and Install Dependencies
import os
import sys
from pathlib import Path

import torch
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image as IPImage

# Configure matplotlib for notebook
%matplotlib inline
matplotlib.rcParams['figure.max_open_warning'] = 50

print(f"Python: {sys.version}")
print(f"Torch version: {torch.__version__}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

IS_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')
if IS_COLAB:
    # Running in Google Colab
    BASE_DIR = Path('/computer_vision_yolo')
else:
    # Running locally
    BASE_DIR = Path.cwd().parent


PROJECT_ROOT = BASE_DIR
SCRIPT_PATH = PROJECT_ROOT / "yolo_test" / "run_yolo_validation_report.py"

# Add project root to path for imports
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(PROJECT_ROOT / "yolo_test") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "yolo_test"))

print(f"Project root: {PROJECT_ROOT}")
print(f"Script path: {SCRIPT_PATH}")


# Dataset Selection
# Option 1: Full dataset (~100k images) - for final optimization: "bdd100k_yolo"
# Option 2: Limited dataset (representative samples) - for quick tuning: "bdd100k_yolo_limited"
DATASET_NAME = 'bdd100k_yolo'
DATASET_SPLT = 'test'  # 'train', 'val', or 'test'

BATCH_SIZE = 256
import sys
sys.path.append("/computer_vision_yolo/yolo_test")
# 2. Import validation functions from script
from run_yolo_validation_report import run_validation_pipeline, visualize_predictions
print("✓ Successfully imported validation functions")

# 3. Method Loop over models, run validation, and collect metrics
results_summary = []
validation_results = {}

def test_model(models_configs):

    for cfg in models_configs:
        print("=" * 80)
        print(f"Running model: {cfg['name']} | dataset={DATASET_NAME} | split={DATASET_SPLT} | IoU={cfg['iou']}")
        print("=" * 80)

        try:
            result = run_validation_pipeline(
                model_name=cfg["name"],
                dataset_name=DATASET_NAME,
                split=DATASET_SPLT,
                iou_threshold=cfg["iou"],
                base_dir=PROJECT_ROOT,
                use_wandb=True,
                save_reports=True,
                batch_size=BATCH_SIZE,
            )

            validation_results[cfg["name"]] = result

            overall = result["metrics"]["overall"]
            yolo_overall = result["metrics"]["yolo_metrics"]

            results_summary.append({
                "model_name": cfg["name"],
                "dataset": DATASET_NAME,
                "split": DATASET_SPLT,
                "iou": cfg["iou"],
                "precision_confusion": overall["precision"],
                "recall_confusion": overall["recall"],
                "f1_confusion": overall["f1"],
                "precision_yolo": yolo_overall["precision"],
                "recall_yolo": yolo_overall["recall"],
                "map50": yolo_overall["map50"],
                "map50_95": yolo_overall["map50_95"],
                "params_m": result["model_info"]["params"] / 1e6,
                "size_mb": result["model_info"]["size(MB)"],
                "fps": result["metrics"]["fps"],
                "status": "ok",
                "run_dir": str(result["run_dir"]),
            })

        except Exception as e:
            print(f"⚠️ Model {cfg['name']} failed: {e}")
            results_summary.append({
                "model_name": cfg["name"],
                "dataset": cfg["dataset"],
                "split": cfg["split"],
                "iou": cfg["iou"],
                "status": "error",
            })

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Torch version: 2.9.0+cu126
Device: cuda
Project root: /computer_vision_yolo
Script path: /computer_vision_yolo/yolo_test/run_yolo_validation_report.py
✓ Successfully imported validation functions


In [ ]:
import os, signal
os.kill(os.getpid(), signal.SIGKILL)


In [ ]:
# 4. Select model configurations to test

MODEL_CONFIGS = [
    {"name": "yolov8n",  "iou": 0.5},
    {"name": "yolov8s",  "iou": 0.5},
    {"name": "yolov8m",  "iou": 0.5},
    {"name": "yolov8l",  "iou": 0.5},
    {"name": "yolov8x",  "iou": 0.5},
]


test_model(MODEL_CONFIGS)


Running model: yolov8n | dataset=bdd100k_yolo | split=test | IoU=0.5
✓ Device: cuda
  GPU: Tesla T4
✓ W&B logging enabled



✓ Weights & Biases initialized: yolov8n_bdd100k_yolo_test_20251123_223435
✓ Dataset loaded
  Total images: 20000
  Images with labels: 20000
  Label files: 20000

✓ Performance metadata loaded: test_performance_analysis.json
  Images with attributes: 20000
test
✓ Model loaded from /computer_vision_yolo/models/yolov8n/yolov8n.pt
YOLOv8n summary: 129 layers, 3,157,200 parameters, 0 gradients, 8.9 GFLOPs

📊 Model Information:
  Model: yolov8n
  Classes in model: 80
  Task: detect
  Parameters: 3.2M
  Model Size: 6.2 MB
  FLOPs (640x640): 8.86 GFLOPs
  Model Size: 6.2 MB

Running YOLO validation...
Ultralytics 8.3.231 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 983.7±584.2 MB/s, size: 56.7 KB)
val: Scanning /computer_vision_yolo/bdd100k_yolo/labels/test.cache... 20000 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 20000/20000 18.2Mit/s

Generating comparisons: 100%|██████████| 6/6 [00:14<00:00,  2.41s/it]


✓ Generated 6 comparison images
  Saved to: /computer_vision_yolo/yolo_test/runs/yolov8n_testing_20251123_223435/sample_comparisons
✓ COMPREHENSIVE REPORT GENERATED (script)
PDF Report: /computer_vision_yolo/yolo_test/runs/yolov8n_testing_20251123_223435/report.pdf
JSON Metrics: /computer_vision_yolo/yolo_test/runs/yolov8n_testing_20251123_223435/metrics_data.json



✓ Weights & Biases run completed successfully
Running model: yolov8s | dataset=bdd100k_yolo | split=test | IoU=0.5
✓ Device: cuda
  GPU: Tesla T4
✓ W&B logging enabled


...

Generating comparisons: 100%|██████████| 6/6 [00:15<00:00,  2.54s/it]


✓ Generated 6 comparison images
  Saved to: /computer_vision_yolo/yolo_test/runs/yolov8m_testing_20251123_224305/sample_comparisons
✓ COMPREHENSIVE REPORT GENERATED (script)
PDF Report: /computer_vision_yolo/yolo_test/runs/yolov8m_testing_20251123_224305/report.pdf
JSON Metrics: /computer_vision_yolo/yolo_test/runs/yolov8m_testing_20251123_224305/metrics_data.json



✓ Weights & Biases run completed successfully
Running model: yolov8l | dataset=bdd100k_yolo | split=test | IoU=0.5
✓ Device: cuda
  GPU: Tesla T4
✓ W&B logging enabled



✓ Weights & Biases initialized: yolov8l_bdd100k_yolo_test_20251123_225012
✓ Dataset loaded
  Total images: 20000
  Images with labels: 20000
  Label files: 20000

✓ Performance metadata loaded: test_performance_analysis.json
  Images with attributes: 20000
Model not found at /computer_vision_yolo/models/yolov8l/yolov8l.pt
✓ Model downloaded and saved to /computer_vision_yolo/models/yolov8l/yolov8l.pt
  Size: 83.7 MB
YOLOv8l summary: 209 layers, 43,691,520 parameters, 0 gradients, 165.7 GFLOPs

📊 Model Information:
  Model: yolov8l
  Classes in model: 80
  Task: detect
  Parameters: 43.7M
  Model Size: 83.7 MB
  FLOPs (640x640): 165.74 GFLOPs
  Model Size: 83.7 MB

Running YOLO validation...
Ultralytics 8.3.231 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLOv8l summary (fused): 112 layers, 43,668,288 parameters, 0 gradients, 165.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 848.1±372.5 MB/s, size: 55.1 KB)
val: Scanning /computer_vision_yolo/bdd100k_yol

KeyError: 'dataset'

In [ ]:
BATCH_SIZE = 128

MODEL_CONFIGS = [
  {"name": "yolov8l",  "iou": 0.5},
  {"name": "yolov8x",  "iou": 0.5},

]

test_model(MODEL_CONFIGS)


Running model: yolov8l | dataset=bdd100k_yolo | split=test | IoU=0.5
✓ Device: cuda
  GPU: Tesla T4
✓ W&B logging enabled


wandb: Currently logged in as: m3mahdy (m3mahdy-king-saud-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



✓ Weights & Biases initialized: yolov8l_bdd100k_yolo_test_20251123_225452
✓ Dataset loaded
  Total images: 20000
  Images with labels: 20000
  Label files: 20000

✓ Performance metadata loaded: test_performance_analysis.json
  Images with attributes: 20000
test
✓ Model loaded from /computer_vision_yolo/models/yolov8l/yolov8l.pt
YOLOv8l summary: 209 layers, 43,691,520 parameters, 0 gradients, 165.7 GFLOPs

📊 Model Information:
  Model: yolov8l
  Classes in model: 80
  Task: detect
  Parameters: 43.7M
  Model Size: 83.7 MB
  FLOPs (640x640): 165.74 GFLOPs
  Model Size: 83.7 MB

Running YOLO validation...
Ultralytics 8.3.231 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLOv8l summary (fused): 112 layers, 43,668,288 parameters, 0 gradients, 165.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 629.2±678.7 MB/s, size: 48.7 KB)
val: Scanning /computer_vision_yolo/bdd100k_yolo/labels/test.cache... 20000 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 20000/200

Generating comparisons: 100%|██████████| 6/6 [00:15<00:00,  2.60s/it]


✓ Generated 6 comparison images
  Saved to: /computer_vision_yolo/yolo_test/runs/yolov8l_testing_20251123_225452/sample_comparisons
✓ COMPREHENSIVE REPORT GENERATED (script)
PDF Report: /computer_vision_yolo/yolo_test/runs/yolov8l_testing_20251123_225452/report.pdf
JSON Metrics: /computer_vision_yolo/yolo_test/runs/yolov8l_testing_20251123_225452/metrics_data.json



✓ Weights & Biases run completed successfully
Running model: yolov8x | dataset=bdd100k_yolo | split=test | IoU=0.5
✓ Device: cuda
  GPU: Tesla T4
✓ W&B logging enabled



✓ Weights & Biases initialized: yolov8x_bdd100k_yolo_test_20251123_230505
✓ Dataset loaded
  Total images: 20000
  Images with labels: 20000
  Label files: 20000

✓ Performance metadata loaded: test_performance_analysis.json
  Images with attributes: 20000
Model not found at /computer_vision_yolo/models/yolov8x/yolov8x.pt
✓ Model downloaded and saved to /computer_vision_yolo/models/yolov8x/yolov8x.pt
  Size: 130.5 MB
YOLOv8x summary: 209 layers, 68,229,648 parameters, 0 gradients, 258.5 GFLOPs

📊 Model Information:
  Model: yolov8x
  Classes in model: 80
  Task: detect
  Parameters: 68.2M
  Model Size: 130.5 MB
  FLOPs (640x640): 258.55 GFLOPs
  Model Size: 130.5 MB

Running YOLO validation...
Ultralytics 8.3.231 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLOv8x summary (fused): 112 layers, 68,200,608 parameters, 0 gradients, 257.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1309.0±489.2 MB/s, size: 63.4 KB)
val: Scanning /computer_vision_yolo/bdd100k

Generating comparisons: 100%|██████████| 6/6 [00:14<00:00,  2.44s/it]


✓ Generated 6 comparison images
  Saved to: /computer_vision_yolo/yolo_test/runs/yolov8x_testing_20251123_230505/sample_comparisons
✓ COMPREHENSIVE REPORT GENERATED (script)
PDF Report: /computer_vision_yolo/yolo_test/runs/yolov8x_testing_20251123_230505/report.pdf
JSON Metrics: /computer_vision_yolo/yolo_test/runs/yolov8x_testing_20251123_230505/metrics_data.json



✓ Weights & Biases run completed successfully


In [ ]:
BATCH_SIZE = 32

MODEL_CONFIGS = [
  {"name": "yolov9e",  "iou": 0.5},
]

test_model(MODEL_CONFIGS)


Running model: yolov9e | dataset=bdd100k_yolo | split=test | IoU=0.5
✓ Device: cuda
  GPU: Tesla T4
✓ W&B logging enabled


wandb: Currently logged in as: m3mahdy (m3mahdy-king-saud-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



✓ Weights & Biases initialized: yolov9e_bdd100k_yolo_test_20251124_014052
✓ Dataset loaded
  Total images: 20000
  Images with labels: 20000
  Label files: 20000

✓ Performance metadata loaded: test_performance_analysis.json
  Images with attributes: 20000
test
✓ Model loaded from /computer_vision_yolo/models/yolov9e/yolov9e.pt
YOLOv9e summary: 721 layers, 58,206,592 parameters, 0 gradients, 193.0 GFLOPs

📊 Model Information:
  Model: yolov9e
  Classes in model: 80
  Task: detect
  Parameters: 58.2M
  Model Size: 112.1 MB
  FLOPs (640x640): 193.02 GFLOPs
  Model Size: 112.1 MB

Running YOLO validation...
Ultralytics 8.3.231 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLOv9e summary (fused): 279 layers, 57,438,080 parameters, 0 gradients, 189.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1351.4±547.4 MB/s, size: 60.9 KB)
val: Scanning /computer_vision_yolo/bdd100k_yolo/labels/test.cache... 20000 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 20000/

Generating comparisons: 100%|██████████| 6/6 [00:14<00:00,  2.45s/it]


✓ Generated 6 comparison images
  Saved to: /computer_vision_yolo/yolo_test/runs/yolov9e_testing_20251124_014052/sample_comparisons
✓ COMPREHENSIVE REPORT GENERATED (script)
PDF Report: /computer_vision_yolo/yolo_test/runs/yolov9e_testing_20251124_014052/report.pdf
JSON Metrics: /computer_vision_yolo/yolo_test/runs/yolov9e_testing_20251124_014052/metrics_data.json



✓ Weights & Biases run completed successfully


In [ ]:
BATCH_SIZE = 128

MODEL_CONFIGS = [
  {"name": "yolov9c",  "iou": 0.5},
]

test_model(MODEL_CONFIGS)


Running model: yolov9c | dataset=bdd100k_yolo | split=test | IoU=0.5
✓ Device: cuda
  GPU: Tesla T4
✓ W&B logging enabled


wandb: Currently logged in as: m3mahdy (m3mahdy-king-saud-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



✓ Weights & Biases initialized: yolov9c_bdd100k_yolo_test_20251123_233112
✓ Dataset loaded
  Total images: 20000
  Images with labels: 20000
  Label files: 20000

✓ Performance metadata loaded: test_performance_analysis.json
  Images with attributes: 20000
test
✓ Model loaded from /computer_vision_yolo/models/yolov9c/yolov9c.pt
YOLOv9c summary: 358 layers, 25,590,912 parameters, 0 gradients, 104.0 GFLOPs

📊 Model Information:
  Model: yolov9c
  Classes in model: 80
  Task: detect
  Parameters: 25.6M
  Model Size: 49.4 MB
  FLOPs (640x640): 104.02 GFLOPs
  Model Size: 49.4 MB

Running YOLO validation...
Ultralytics 8.3.231 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLOv9c summary (fused): 156 layers, 25,380,928 parameters, 0 gradients, 102.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1701.1±291.6 MB/s, size: 50.3 KB)
val: Scanning /computer_vision_yolo/bdd100k_yolo/labels/test.cache... 20000 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 20000/20

Generating comparisons: 100%|██████████| 6/6 [00:14<00:00,  2.47s/it]


✓ Generated 6 comparison images
  Saved to: /computer_vision_yolo/yolo_test/runs/yolov9c_testing_20251123_233112/sample_comparisons
✓ COMPREHENSIVE REPORT GENERATED (script)
PDF Report: /computer_vision_yolo/yolo_test/runs/yolov9c_testing_20251123_233112/report.pdf
JSON Metrics: /computer_vision_yolo/yolo_test/runs/yolov9c_testing_20251123_233112/metrics_data.json



✓ Weights & Biases run completed successfully


In [ ]:
BATCH_SIZE = 128

MODEL_CONFIGS = [

  {"name": "yolov10x",  "iou": 0.5},


]

test_model(MODEL_CONFIGS)

Running model: yolov10x | dataset=bdd100k_yolo | split=test | IoU=0.5
✓ Device: cuda
  GPU: Tesla T4
✓ W&B logging enabled



✓ Weights & Biases initialized: yolov10x_bdd100k_yolo_test_20251123_234252
✓ Dataset loaded
  Total images: 20000
  Images with labels: 20000
  Label files: 20000

✓ Performance metadata loaded: test_performance_analysis.json
  Images with attributes: 20000
Model not found at /computer_vision_yolo/models/yolov10x/yolov10x.pt
✓ Model downloaded and saved to /computer_vision_yolo/models/yolov10x/yolov10x.pt
  Size: 61.4 MB
YOLOv10x summary: 400 layers, 31,808,960 parameters, 0 gradients, 171.8 GFLOPs

📊 Model Information:
  Model: yolov10x
  Classes in model: 80
  Task: detect
  Parameters: 31.8M
  Model Size: 61.4 MB
  FLOPs (640x640): 171.85 GFLOPs
  Model Size: 61.4 MB

Running YOLO validation...
Ultralytics 8.3.231 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLOv10x summary (fused): 192 layers, 29,473,568 parameters, 0 gradients, 160.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1123.3±391.8 MB/s, size: 56.2 KB)
val: Scanning /computer_vision_yolo/bd

Generating comparisons: 100%|██████████| 6/6 [00:14<00:00,  2.37s/it]


✓ Generated 6 comparison images
  Saved to: /computer_vision_yolo/yolo_test/runs/yolov10x_testing_20251123_234252/sample_comparisons
✓ COMPREHENSIVE REPORT GENERATED (script)
PDF Report: /computer_vision_yolo/yolo_test/runs/yolov10x_testing_20251123_234252/report.pdf
JSON Metrics: /computer_vision_yolo/yolo_test/runs/yolov10x_testing_20251123_234252/metrics_data.json



✓ Weights & Biases run completed successfully


In [ ]:
BATCH_SIZE = 128

MODEL_CONFIGS = [

  {"name": "yolo11x",  "iou": 0.5},
]

test_model(MODEL_CONFIGS)

Running model: yolo11x | dataset=bdd100k_yolo | split=test | IoU=0.5
✓ Device: cuda
  GPU: Tesla T4
✓ W&B logging enabled


wandb: Currently logged in as: m3mahdy (m3mahdy-king-saud-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



✓ Weights & Biases initialized: yolo11x_bdd100k_yolo_test_20251124_000753
✓ Dataset loaded
  Total images: 20000
  Images with labels: 20000
  Label files: 20000

✓ Performance metadata loaded: test_performance_analysis.json
  Images with attributes: 20000
test
✓ Model loaded from /computer_vision_yolo/models/yolo11x/yolo11x.pt
YOLO11x summary: 357 layers, 56,966,176 parameters, 0 gradients, 196.0 GFLOPs

📊 Model Information:
  Model: yolo11x
  Classes in model: 80
  Task: detect
  Parameters: 57.0M
  Model Size: 109.3 MB
  FLOPs (640x640): 195.96 GFLOPs
  Model Size: 109.3 MB

Running YOLO validation...
Ultralytics 8.3.231 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11x summary (fused): 190 layers, 56,919,424 parameters, 0 gradients, 194.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1321.5±561.0 MB/s, size: 49.9 KB)
val: Scanning /computer_vision_yolo/bdd100k_yolo/labels/test.cache... 20000 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 20000/

Generating comparisons: 100%|██████████| 6/6 [00:14<00:00,  2.38s/it]


✓ Generated 6 comparison images
  Saved to: /computer_vision_yolo/yolo_test/runs/yolo11x_testing_20251124_000753/sample_comparisons
✓ COMPREHENSIVE REPORT GENERATED (script)
PDF Report: /computer_vision_yolo/yolo_test/runs/yolo11x_testing_20251124_000753/report.pdf
JSON Metrics: /computer_vision_yolo/yolo_test/runs/yolo11x_testing_20251124_000753/metrics_data.json



✓ Weights & Biases run completed successfully


In [ ]:
BATCH_SIZE = 128

MODEL_CONFIGS = [
  {"name": "yolo12x",  "iou": 0.5},
]

test_model(MODEL_CONFIGS)

Running model: yolo12x | dataset=bdd100k_yolo | split=test | IoU=0.5
✓ Device: cuda
  GPU: Tesla T4
✓ W&B logging enabled


wandb: Currently logged in as: m3mahdy (m3mahdy-king-saud-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



✓ Weights & Biases initialized: yolo12x_bdd100k_yolo_test_20251124_005556
✓ Dataset loaded
  Total images: 20000
  Images with labels: 20000
  Label files: 20000

✓ Performance metadata loaded: test_performance_analysis.json
  Images with attributes: 20000
Model not found at /computer_vision_yolo/models/yolo12x/yolo12x.pt
✓ Model downloaded and saved to /computer_vision_yolo/models/yolo12x/yolo12x.pt
  Size: 113.8 MB
YOLOv12x summary: 488 layers, 59,216,928 parameters, 0 gradients, 200.3 GFLOPs

📊 Model Information:
  Model: yolo12x
  Classes in model: 80
  Task: detect
  Parameters: 59.2M
  Model Size: 113.8 MB
  FLOPs (640x640): 200.33 GFLOPs
  Model Size: 113.8 MB

Running YOLO validation...
Ultralytics 8.3.231 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLOv12x summary (fused): 283 layers, 59,135,744 parameters, 0 gradients, 199.0 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1655.5±685.5 MB/s, size: 57.3 KB)
val: Scanning /computer_vision_yolo/bdd10

Generating comparisons: 100%|██████████| 6/6 [00:14<00:00,  2.36s/it]


✓ Generated 6 comparison images
  Saved to: /computer_vision_yolo/yolo_test/runs/yolo12x_testing_20251124_005556/sample_comparisons
✓ COMPREHENSIVE REPORT GENERATED (script)
PDF Report: /computer_vision_yolo/yolo_test/runs/yolo12x_testing_20251124_005556/report.pdf
JSON Metrics: /computer_vision_yolo/yolo_test/runs/yolo12x_testing_20251124_005556/metrics_data.json



✓ Weights & Biases run completed successfully


In [ ]:
BATCH_SIZE = 256

MODEL_CONFIGS = [
  {"name": "yolov9t",  "iou": 0.5},
  {"name": "yolov9s",  "iou": 0.5},
]

test_model(MODEL_CONFIGS)

Running model: yolov9t | dataset=bdd100k_yolo | split=test | IoU=0.5
✓ Device: cuda
  GPU: Tesla T4
✓ W&B logging enabled


wandb: Currently logged in as: m3mahdy (m3mahdy-king-saud-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



✓ Weights & Biases initialized: yolov9t_bdd100k_yolo_test_20251124_002542
✓ Dataset loaded
  Total images: 20000
  Images with labels: 20000
  Label files: 20000

✓ Performance metadata loaded: test_performance_analysis.json
  Images with attributes: 20000
Model not found at /computer_vision_yolo/models/yolov9t/yolov9t.pt
✓ Model downloaded and saved to /computer_vision_yolo/models/yolov9t/yolov9t.pt
  Size: 4.7 MB
YOLOv9t summary: 544 layers, 2,128,720 parameters, 0 gradients, 8.5 GFLOPs

📊 Model Information:
  Model: yolov9t
  Classes in model: 80
  Task: detect
  Parameters: 2.1M
  Model Size: 4.7 MB
  FLOPs (640x640): 8.48 GFLOPs
  Model Size: 4.7 MB

Running YOLO validation...
Ultralytics 8.3.231 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLOv9t summary (fused): 197 layers, 2,094,000 parameters, 0 gradients, 8.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1508.4±412.6 MB/s, size: 56.7 KB)
val: Scanning /computer_vision_yolo/bdd100k_yolo/labels/te

Generating comparisons: 100%|██████████| 6/6 [00:14<00:00,  2.35s/it]


✓ Generated 6 comparison images
  Saved to: /computer_vision_yolo/yolo_test/runs/yolov9t_testing_20251124_002542/sample_comparisons
✓ COMPREHENSIVE REPORT GENERATED (script)
PDF Report: /computer_vision_yolo/yolo_test/runs/yolov9t_testing_20251124_002542/report.pdf
JSON Metrics: /computer_vision_yolo/yolo_test/runs/yolov9t_testing_20251124_002542/metrics_data.json



✓ Weights & Biases run completed successfully
Running model: yolov9s | dataset=bdd100k_yolo | split=test | IoU=0.5
✓ Device: cuda
  GPU: Tesla T4
✓ W&B logging enabled



✓ Weights & Biases initialized: yolov9s_bdd100k_yolo_test_20251124_002940
✓ Dataset loaded
  Total images: 20000
  Images with labels: 20000
  Label files: 20000

✓ Performance metadata loaded: test_performance_analysis.json
  Images with attributes: 20000
test
✓ Model loaded from /computer_vision_yolo/models/yolov9s/yolov9s.pt
YOLOv9s summary: 544 layers, 7,318,368 parameters, 0 gradients, 27.6 GFLOPs

📊 Model Information:
  Model: yolov9s
  Classes in model: 80
  Task: detect
  Parameters: 7.3M
  Model Size: 14.7 MB
  FLOPs (640x640): 27.56 GFLOPs
  Model Size: 14.7 MB

Running YOLO validation...
Ultralytics 8.3.231 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLOv9s summary (fused): 197 layers, 7,198,048 parameters, 0 gradients, 26.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1470.9±458.8 MB/s, size: 69.0 KB)
val: Scanning /computer_vision_yolo/bdd100k_yolo/labels/test.cache... 20000 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 20000/20000 11

Generating comparisons: 100%|██████████| 6/6 [00:13<00:00,  2.32s/it]


✓ Generated 6 comparison images
  Saved to: /computer_vision_yolo/yolo_test/runs/yolov9s_testing_20251124_002940/sample_comparisons
✓ COMPREHENSIVE REPORT GENERATED (script)
PDF Report: /computer_vision_yolo/yolo_test/runs/yolov9s_testing_20251124_002940/report.pdf
JSON Metrics: /computer_vision_yolo/yolo_test/runs/yolov9s_testing_20251124_002940/metrics_data.json



✓ Weights & Biases run completed successfully


In [ ]:
BATCH_SIZE = 256

MODEL_CONFIGS = [
  {"name": "yolov10n",  "iou": 0.5},

]

test_model(MODEL_CONFIGS)

Running model: yolov10n | dataset=bdd100k_yolo | split=test | IoU=0.5
✓ Device: cuda
  GPU: Tesla T4
✓ W&B logging enabled


wandb: Currently logged in as: m3mahdy (m3mahdy-king-saud-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



✓ Weights & Biases initialized: yolov10n_bdd100k_yolo_test_20251124_003508
✓ Dataset loaded
  Total images: 20000
  Images with labels: 20000
  Label files: 20000

✓ Performance metadata loaded: test_performance_analysis.json
  Images with attributes: 20000
test
✓ Model loaded from /computer_vision_yolo/models/yolov10n/yolov10n.pt
YOLOv10n summary: 223 layers, 2,775,520 parameters, 0 gradients, 8.7 GFLOPs

📊 Model Information:
  Model: yolov10n
  Classes in model: 80
  Task: detect
  Parameters: 2.8M
  Model Size: 5.6 MB
  FLOPs (640x640): 8.74 GFLOPs
  Model Size: 5.6 MB

Running YOLO validation...
Ultralytics 8.3.231 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLOv10n summary (fused): 102 layers, 2,299,264 parameters, 0 gradients, 6.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1478.9±558.8 MB/s, size: 51.8 KB)
val: Scanning /computer_vision_yolo/bdd100k_yolo/labels/test.cache... 20000 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 20000/20000 1

Generating comparisons: 100%|██████████| 6/6 [00:14<00:00,  2.36s/it]


✓ Generated 6 comparison images
  Saved to: /computer_vision_yolo/yolo_test/runs/yolov10n_testing_20251124_003508/sample_comparisons
✓ COMPREHENSIVE REPORT GENERATED (script)
PDF Report: /computer_vision_yolo/yolo_test/runs/yolov10n_testing_20251124_003508/report.pdf
JSON Metrics: /computer_vision_yolo/yolo_test/runs/yolov10n_testing_20251124_003508/metrics_data.json



✓ Weights & Biases run completed successfully


In [ ]:
BATCH_SIZE = 256

MODEL_CONFIGS = [

  {"name": "yolov10s",  "iou": 0.5},
]

test_model(MODEL_CONFIGS)

Running model: yolov10s | dataset=bdd100k_yolo | split=test | IoU=0.5
✓ Device: cuda
  GPU: Tesla T4
✓ W&B logging enabled



✓ Weights & Biases initialized: yolov10s_bdd100k_yolo_test_20251124_012004
✓ Dataset loaded
  Total images: 20000
  Images with labels: 20000
  Label files: 20000

✓ Performance metadata loaded: test_performance_analysis.json
  Images with attributes: 20000
test
✓ Model loaded from /computer_vision_yolo/models/yolov10s/yolov10s.pt
YOLOv10s summary: 234 layers, 8,128,272 parameters, 0 gradients, 25.1 GFLOPs

📊 Model Information:
  Model: yolov10s
  Classes in model: 80
  Task: detect
  Parameters: 8.1M
  Model Size: 15.9 MB
  FLOPs (640x640): 25.11 GFLOPs
  Model Size: 15.9 MB

Running YOLO validation...
Ultralytics 8.3.231 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLOv10s summary (fused): 106 layers, 7,248,960 parameters, 0 gradients, 21.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1698.9±664.3 MB/s, size: 58.3 KB)
val: Scanning /computer_vision_yolo/bdd100k_yolo/labels/test.cache... 20000 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 20000/20

Generating comparisons: 100%|██████████| 6/6 [00:13<00:00,  2.29s/it]


✓ Generated 6 comparison images
  Saved to: /computer_vision_yolo/yolo_test/runs/yolov10s_testing_20251124_012004/sample_comparisons
✓ COMPREHENSIVE REPORT GENERATED (script)
PDF Report: /computer_vision_yolo/yolo_test/runs/yolov10s_testing_20251124_012004/report.pdf
JSON Metrics: /computer_vision_yolo/yolo_test/runs/yolov10s_testing_20251124_012004/metrics_data.json



✓ Weights & Biases run completed successfully


In [ ]:
BATCH_SIZE = 256

MODEL_CONFIGS = [
  {"name": "yolo11n",  "iou": 0.5},

]

test_model(MODEL_CONFIGS)

Running model: yolo11n | dataset=bdd100k_yolo | split=test | IoU=0.5
✓ Device: cuda
  GPU: Tesla T4
✓ W&B logging enabled


wandb: Currently logged in as: m3mahdy (m3mahdy-king-saud-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



✓ Weights & Biases initialized: yolo11n_bdd100k_yolo_test_20251124_003927
✓ Dataset loaded
  Total images: 20000
  Images with labels: 20000
  Label files: 20000

✓ Performance metadata loaded: test_performance_analysis.json
  Images with attributes: 20000
test
✓ Model loaded from /computer_vision_yolo/models/yolo11n/yolo11n.pt
YOLO11n summary: 181 layers, 2,624,080 parameters, 0 gradients, 6.6 GFLOPs

📊 Model Information:
  Model: yolo11n
  Classes in model: 80
  Task: detect
  Parameters: 2.6M
  Model Size: 5.4 MB
  FLOPs (640x640): 6.61 GFLOPs
  Model Size: 5.4 MB

Running YOLO validation...
Ultralytics 8.3.231 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11n summary (fused): 100 layers, 2,616,248 parameters, 0 gradients, 6.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1909.6±626.1 MB/s, size: 60.1 KB)
val: Scanning /computer_vision_yolo/bdd100k_yolo/labels/test.cache... 20000 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 20000/20000 11.5Mit

Generating comparisons: 100%|██████████| 6/6 [00:14<00:00,  2.37s/it]


✓ Generated 6 comparison images
  Saved to: /computer_vision_yolo/yolo_test/runs/yolo11n_testing_20251124_003927/sample_comparisons
✓ COMPREHENSIVE REPORT GENERATED (script)
PDF Report: /computer_vision_yolo/yolo_test/runs/yolo11n_testing_20251124_003927/report.pdf
JSON Metrics: /computer_vision_yolo/yolo_test/runs/yolo11n_testing_20251124_003927/metrics_data.json



✓ Weights & Biases run completed successfully


In [ ]:
BATCH_SIZE = 256

MODEL_CONFIGS = [

  {"name": "yolo11s",  "iou": 0.5},
]

test_model(MODEL_CONFIGS)

Running model: yolo11s | dataset=bdd100k_yolo | split=test | IoU=0.5
✓ Device: cuda
  GPU: Tesla T4
✓ W&B logging enabled


wandb: Currently logged in as: m3mahdy (m3mahdy-king-saud-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



✓ Weights & Biases initialized: yolo11s_bdd100k_yolo_test_20251124_012509
✓ Dataset loaded
  Total images: 20000
  Images with labels: 20000
  Label files: 20000

✓ Performance metadata loaded: test_performance_analysis.json
  Images with attributes: 20000
Model not found at /computer_vision_yolo/models/yolo11s/yolo11s.pt
✓ Model downloaded and saved to /computer_vision_yolo/models/yolo11s/yolo11s.pt
  Size: 18.4 MB
YOLO11s summary: 181 layers, 9,458,752 parameters, 0 gradients, 21.7 GFLOPs

📊 Model Information:
  Model: yolo11s
  Classes in model: 80
  Task: detect
  Parameters: 9.5M
  Model Size: 18.4 MB
  FLOPs (640x640): 21.72 GFLOPs
  Model Size: 18.4 MB

Running YOLO validation...
Ultralytics 8.3.231 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11s summary (fused): 100 layers, 9,443,760 parameters, 0 gradients, 21.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2105.8±776.0 MB/s, size: 61.8 KB)
val: Scanning /computer_vision_yolo/bdd100k_yolo/lab

Generating comparisons: 100%|██████████| 6/6 [00:14<00:00,  2.35s/it]


✓ Generated 6 comparison images
  Saved to: /computer_vision_yolo/yolo_test/runs/yolo11s_testing_20251124_012509/sample_comparisons
✓ COMPREHENSIVE REPORT GENERATED (script)
PDF Report: /computer_vision_yolo/yolo_test/runs/yolo11s_testing_20251124_012509/report.pdf
JSON Metrics: /computer_vision_yolo/yolo_test/runs/yolo11s_testing_20251124_012509/metrics_data.json



✓ Weights & Biases run completed successfully


In [ ]:
BATCH_SIZE = 256

MODEL_CONFIGS = [
  {"name": "yolo12n",  "iou": 0.5},

]

test_model(MODEL_CONFIGS)

Running model: yolo12n | dataset=bdd100k_yolo | split=test | IoU=0.5
✓ Device: cuda
  GPU: Tesla T4
✓ W&B logging enabled


wandb: Currently logged in as: m3mahdy (m3mahdy-king-saud-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



✓ Weights & Biases initialized: yolo12n_bdd100k_yolo_test_20251124_004348
✓ Dataset loaded
  Total images: 20000
  Images with labels: 20000
  Label files: 20000

✓ Performance metadata loaded: test_performance_analysis.json
  Images with attributes: 20000
Model not found at /computer_vision_yolo/models/yolo12n/yolo12n.pt
✓ Model downloaded and saved to /computer_vision_yolo/models/yolo12n/yolo12n.pt
  Size: 5.3 MB
YOLOv12n summary: 272 layers, 2,603,056 parameters, 0 gradients, 6.7 GFLOPs

📊 Model Information:
  Model: yolo12n
  Classes in model: 80
  Task: detect
  Parameters: 2.6M
  Model Size: 5.3 MB
  FLOPs (640x640): 6.65 GFLOPs
  Model Size: 5.3 MB

Running YOLO validation...
Ultralytics 8.3.231 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLOv12n summary (fused): 159 layers, 2,590,824 parameters, 0 gradients, 6.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1918.8±456.4 MB/s, size: 62.7 KB)
val: Scanning /computer_vision_yolo/bdd100k_yolo/labels/

Generating comparisons: 100%|██████████| 6/6 [00:14<00:00,  2.40s/it]


✓ Generated 6 comparison images
  Saved to: /computer_vision_yolo/yolo_test/runs/yolo12n_testing_20251124_004348/sample_comparisons
✓ COMPREHENSIVE REPORT GENERATED (script)
PDF Report: /computer_vision_yolo/yolo_test/runs/yolo12n_testing_20251124_004348/report.pdf
JSON Metrics: /computer_vision_yolo/yolo_test/runs/yolo12n_testing_20251124_004348/metrics_data.json



✓ Weights & Biases run completed successfully


In [ ]:
BATCH_SIZE = 256

MODEL_CONFIGS = [

  {"name": "yolo12s",  "iou": 0.5},
]

test_model(MODEL_CONFIGS)

Running model: yolo12s | dataset=bdd100k_yolo | split=test | IoU=0.5
✓ Device: cuda
  GPU: Tesla T4
✓ W&B logging enabled


wandb: Currently logged in as: m3mahdy (m3mahdy-king-saud-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



✓ Weights & Biases initialized: yolo12s_bdd100k_yolo_test_20251124_013002
✓ Dataset loaded
  Total images: 20000
  Images with labels: 20000
  Label files: 20000

✓ Performance metadata loaded: test_performance_analysis.json
  Images with attributes: 20000
Model not found at /computer_vision_yolo/models/yolo12s/yolo12s.pt
✓ Model downloaded and saved to /computer_vision_yolo/models/yolo12s/yolo12s.pt
  Size: 18.1 MB
YOLOv12s summary: 272 layers, 9,285,632 parameters, 0 gradients, 21.7 GFLOPs

📊 Model Information:
  Model: yolo12s
  Classes in model: 80
  Task: detect
  Parameters: 9.3M
  Model Size: 18.1 MB
  FLOPs (640x640): 21.69 GFLOPs
  Model Size: 18.1 MB

Running YOLO validation...
Ultralytics 8.3.231 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLOv12s summary (fused): 159 layers, 9,261,840 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1419.6±531.2 MB/s, size: 41.2 KB)
val: Scanning /computer_vision_yolo/bdd100k_yolo/l

Generating comparisons: 100%|██████████| 6/6 [00:14<00:00,  2.35s/it]


✓ Generated 6 comparison images
  Saved to: /computer_vision_yolo/yolo_test/runs/yolo12s_testing_20251124_013002/sample_comparisons
✓ COMPREHENSIVE REPORT GENERATED (script)
PDF Report: /computer_vision_yolo/yolo_test/runs/yolo12s_testing_20251124_013002/report.pdf
JSON Metrics: /computer_vision_yolo/yolo_test/runs/yolo12s_testing_20251124_013002/metrics_data.json



✓ Weights & Biases run completed successfully


In [2]:
BATCH_SIZE = 256

MODEL_CONFIGS = [
  {"name": "yolov9m",  "iou": 0.5},

]

test_model(MODEL_CONFIGS)

Running model: yolov9m | dataset=bdd100k_yolo | split=test | IoU=0.5
✓ Device: cuda
  GPU: Tesla T4
✓ W&B logging enabled


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: m3mahdy (m3mahdy-king-saud-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



✓ Weights & Biases initialized: yolov9m_bdd100k_yolo_test_20251124_094248
✓ Dataset loaded
  Total images: 20000
  Images with labels: 20000
  Label files: 20000

✓ Performance metadata loaded: test_performance_analysis.json
  Images with attributes: 20000
test
✓ Model loaded from /computer_vision_yolo/models/yolov9m/yolov9m.pt
YOLOv9m summary: 348 layers, 20,216,160 parameters, 0 gradients, 77.9 GFLOPs

📊 Model Information:
  Model: yolov9m
  Classes in model: 80
  Task: detect
  Parameters: 20.2M
  Model Size: 39.1 MB
  FLOPs (640x640): 77.87 GFLOPs
  Model Size: 39.1 MB

Running YOLO validation...
Ultralytics 8.3.231 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLOv9m summary (fused): 151 layers, 20,070,832 parameters, 0 gradients, 76.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 694.1±535.1 MB/s, size: 34.1 KB)
val: Scanning /computer_vision_yolo/bdd100k_yolo/labels/test... 20000 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 20000/20000 979.3i

Generating comparisons: 100%|██████████| 6/6 [00:13<00:00,  2.29s/it]


✓ Generated 6 comparison images
  Saved to: /computer_vision_yolo/yolo_test/runs/yolov9m_testing_20251124_094248/sample_comparisons
✓ COMPREHENSIVE REPORT GENERATED (script)
PDF Report: /computer_vision_yolo/yolo_test/runs/yolov9m_testing_20251124_094248/report.pdf
JSON Metrics: /computer_vision_yolo/yolo_test/runs/yolov9m_testing_20251124_094248/metrics_data.json



✓ Weights & Biases run completed successfully

🧹 Cleaning up model from memory...
✓ Model removed from memory


In [3]:
BATCH_SIZE = 256

MODEL_CONFIGS = [
  {"name": "yolov10m",  "iou": 0.5},

]

test_model(MODEL_CONFIGS)

Running model: yolov10m | dataset=bdd100k_yolo | split=test | IoU=0.5
✓ Device: cuda
  GPU: Tesla T4
✓ W&B logging enabled



✓ Weights & Biases initialized: yolov10m_bdd100k_yolo_test_20251124_095059
✓ Dataset loaded
  Total images: 20000
  Images with labels: 20000
  Label files: 20000

✓ Performance metadata loaded: test_performance_analysis.json
  Images with attributes: 20000
test
✓ Model loaded from /computer_vision_yolo/models/yolov10m/yolov10m.pt
YOLOv10m summary: 288 layers, 16,576,768 parameters, 0 gradients, 64.5 GFLOPs

📊 Model Information:
  Model: yolov10m
  Classes in model: 80
  Task: detect
  Parameters: 16.6M
  Model Size: 32.1 MB
  FLOPs (640x640): 64.48 GFLOPs
  Model Size: 32.1 MB

Running YOLO validation...
Ultralytics 8.3.231 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLOv10m summary (fused): 136 layers, 15,359,488 parameters, 0 gradients, 59.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 977.1±250.4 MB/s, size: 49.5 KB)
val: Scanning /computer_vision_yolo/bdd100k_yolo/labels/test.cache... 20000 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 20000/

Generating comparisons: 100%|██████████| 6/6 [00:12<00:00,  2.16s/it]


✓ Generated 6 comparison images
  Saved to: /computer_vision_yolo/yolo_test/runs/yolov10m_testing_20251124_095059/sample_comparisons
✓ COMPREHENSIVE REPORT GENERATED (script)
PDF Report: /computer_vision_yolo/yolo_test/runs/yolov10m_testing_20251124_095059/report.pdf
JSON Metrics: /computer_vision_yolo/yolo_test/runs/yolov10m_testing_20251124_095059/metrics_data.json



✓ Weights & Biases run completed successfully

🧹 Cleaning up model from memory...
✓ Model removed from memory


In [2]:
BATCH_SIZE = 128

MODEL_CONFIGS = [
  {"name": "yolo11m",  "iou": 0.5},

]

test_model(MODEL_CONFIGS)

Running model: yolo11m | dataset=bdd100k_yolo | split=test | IoU=0.5
✓ Device: cuda
  GPU: Tesla T4
✓ W&B logging enabled


wandb: Currently logged in as: m3mahdy (m3mahdy-king-saud-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



✓ Weights & Biases initialized: yolo11m_bdd100k_yolo_test_20251124_100013
✓ Dataset loaded
  Total images: 20000
  Images with labels: 20000
  Label files: 20000

✓ Performance metadata loaded: test_performance_analysis.json
  Images with attributes: 20000
test
✓ Model loaded from /computer_vision_yolo/models/yolo11m/yolo11m.pt
YOLO11m summary: 231 layers, 20,114,688 parameters, 0 gradients, 68.5 GFLOPs

📊 Model Information:
  Model: yolo11m
  Classes in model: 80
  Task: detect
  Parameters: 20.1M
  Model Size: 38.8 MB
  FLOPs (640x640): 68.53 GFLOPs
  Model Size: 38.8 MB

Running YOLO validation...
Ultralytics 8.3.231 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11m summary (fused): 125 layers, 20,091,712 parameters, 0 gradients, 68.0 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1181.0±296.4 MB/s, size: 44.7 KB)
val: Scanning /computer_vision_yolo/bdd100k_yolo/labels/test.cache... 20000 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 20000/20000

Generating comparisons: 100%|██████████| 6/6 [00:13<00:00,  2.31s/it]


✓ Generated 6 comparison images
  Saved to: /computer_vision_yolo/yolo_test/runs/yolo11m_testing_20251124_100013/sample_comparisons
✓ COMPREHENSIVE REPORT GENERATED (script)
PDF Report: /computer_vision_yolo/yolo_test/runs/yolo11m_testing_20251124_100013/report.pdf
JSON Metrics: /computer_vision_yolo/yolo_test/runs/yolo11m_testing_20251124_100013/metrics_data.json



✓ Weights & Biases run completed successfully

🧹 Cleaning up model from memory...
✓ Model removed from memory


In [2]:
BATCH_SIZE = 128

MODEL_CONFIGS = [
  {"name": "yolo12m",  "iou": 0.5},

]

test_model(MODEL_CONFIGS)

Running model: yolo12m | dataset=bdd100k_yolo | split=test | IoU=0.5
✓ Device: cuda
  GPU: Tesla T4
✓ W&B logging enabled


wandb: Currently logged in as: m3mahdy (m3mahdy-king-saud-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



✓ Weights & Biases initialized: yolo12m_bdd100k_yolo_test_20251124_101029
✓ Dataset loaded
  Total images: 20000
  Images with labels: 20000
  Label files: 20000

✓ Performance metadata loaded: test_performance_analysis.json
  Images with attributes: 20000
test
✓ Model loaded from /computer_vision_yolo/models/yolo12m/yolo12m.pt
YOLOv12m summary: 292 layers, 20,201,216 parameters, 0 gradients, 68.1 GFLOPs

📊 Model Information:
  Model: yolo12m
  Classes in model: 80
  Task: detect
  Parameters: 20.2M
  Model Size: 39.0 MB
  FLOPs (640x640): 68.08 GFLOPs
  Model Size: 39.0 MB

Running YOLO validation...
Ultralytics 8.3.231 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLOv12m summary (fused): 169 layers, 20,166,592 parameters, 0 gradients, 67.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1910.2±714.7 MB/s, size: 57.7 KB)
val: Scanning /computer_vision_yolo/bdd100k_yolo/labels/test.cache... 20000 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 20000/200

Generating comparisons: 100%|██████████| 6/6 [00:13<00:00,  2.32s/it]


✓ Generated 6 comparison images
  Saved to: /computer_vision_yolo/yolo_test/runs/yolo12m_testing_20251124_101029/sample_comparisons
✓ COMPREHENSIVE REPORT GENERATED (script)
PDF Report: /computer_vision_yolo/yolo_test/runs/yolo12m_testing_20251124_101029/report.pdf
JSON Metrics: /computer_vision_yolo/yolo_test/runs/yolo12m_testing_20251124_101029/metrics_data.json



✓ Weights & Biases run completed successfully

🧹 Cleaning up model from memory...
✓ Model removed from memory
